In [1]:
def read_dataframe(filename):
    columns = [
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "trip_distance"
    ]

    df = pd.read_parquet(filename, columns=columns)
    df=df.head(1000)  # For testing purposes, limit to first 1000 rows
    df["duration"] = (
        df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    ).dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ["PULocationID", "DOLocationID"]
    numerical = ["trip_distance"]

    df[categorical] = df[categorical].astype(str)

    return df

In [2]:
import numpy as np
import pickle

In [4]:
import pandas as pd
import sklearn
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import Lasso
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("new")

<Experiment: artifact_location='/workspaces/MLOPs/01-intro/mlruns/1', creation_time=1789148677294, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1789148677294, lifecycle_stage='active', name='new', tags={}, trace_location=None, workspace='default'>

In [5]:
df_train=read_dataframe("./yellow_tripdata_2026-01.parquet")

In [6]:
categorical=["PULocationID","DOLocationID"]
numerical=["trip_distance"]
dv=DictVectorizer()
train_dict=df_train[categorical+numerical].to_dict(orient="records")
X_train=dv.fit_transform(train_dict)


In [7]:
y_train=df_train["duration"]

In [8]:
X_train.indices = X_train.indices.astype(np.int32)
X_train.indptr = X_train.indptr.astype(np.int32)

In [9]:
with mlflow.start_run():
    mlflow.set_tag("developper","hakim")
    alpha=.002
    mlflow.log_param("alpha",alpha)
    lr=Lasso(alpha=alpha)
    lr.fit(X_train,y_train)
    y_pred=lr.predict(X_train)
    mae=mean_absolute_error(y_train,y_pred)
    mlflow.log_metric("mae",mae)

In [10]:
with open("../models/lin_reg.bin","wb") as f_out:
    pickle.dump((dv,lr),f_out)

In [11]:
import xgboost as xgb
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

In [16]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model","xgboost")
        mlflow.log_params(params)

        booster = xgb.train(
            params=params,
            dtrain=xgb.DMatrix(X_train, label=y_train),
            num_boost_round=100,
            evals=[(xgb.DMatrix(X_train, label=y_train), "train")],
            early_stopping_rounds=50
        )
        y_pred = booster.predict(xgb.DMatrix(X_train))
        mae = mean_absolute_error(y_train, y_pred)


        mlflow.log_metric("mae", mae)

    return {"loss": mae, "status": STATUS_OK}

In [17]:
search_space = {
    "max_depth": scope.int(hp.quniform("max_depth", 4, 100, 1)),
    "learning_rate": hp.loguniform("learning_rate", -3, 0),
    "reg_alpha": hp.loguniform("reg_alpha", -5, -1),
    "reg_lambda": hp.loguniform("reg_lambda", -6, -1),
    "min_child_weight": hp.loguniform("min_child_weight", -1, 3),
    "objective": "reg:squarederror",
}
best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=Trials(),
)

[0]	train-rmse:9.45582                                
[1]	train-rmse:8.56899                                
[2]	train-rmse:7.87683                                
[3]	train-rmse:7.33793                                
[4]	train-rmse:6.92668                                
[5]	train-rmse:6.60668                                
[6]	train-rmse:6.33191                                
[7]	train-rmse:6.08930                                
[8]	train-rmse:5.90957                                
[9]	train-rmse:5.73344                                
[10]	train-rmse:5.61081                               
[11]	train-rmse:5.47851                               
[12]	train-rmse:5.33655                               
[13]	train-rmse:5.22507                               
[14]	train-rmse:5.13084                               
[15]	train-rmse:5.06233                               
[16]	train-rmse:4.99967                               
[17]	train-rmse:4.91771                               
[18]	train